In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

In [ ]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

# **1. Profile the data — find the problems**

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts


In [ ]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")


In [ ]:

# 1. Duplicate row count
print("Duplicate rows:", df.duplicated().sum())

# 2. Missing count per column
print("\nMissing values per column:")
print(df.isnull().sum())

# 3. Problems I can see:
# - There may be duplicate rows.
# - Some columns may contain missing values.
# - Some columns may have incorrect or inconsistent data types.
# - There may be inconsistent formatting or invalid values.

# 2. Missing values — detect & **handle**

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())



In [ ]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())


In [ ]:
import numpy as np
import pandas as pd

ex = raw.copy()

# 1. Unmask missing values
ex["spend"] = pd.to_numeric(ex["spend"], errors="coerce")
ex["age"] = ex["age"].replace(-1, np.nan)

# 2a. Dropna version
drop_version = ex.dropna()

# 2b. Median-impute version
impute_version = ex.copy()

impute_version["age"] = impute_version["age"].fillna(
    impute_version["age"].median()
)

impute_version["spend"] = impute_version["spend"].fillna(
    impute_version["spend"].median()
)

# 3. Compare row counts
print("Original rows:", len(ex))
print("After dropna:", len(drop_version))
print("After median imputation:", len(impute_version))
print("Rows lost by dropping:", len(ex) - len(drop_version))

# 3. Duplicates & data **types**

In [ ]:
 -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')


In [ ]:
# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)


In [ ]:
import pandas as pd

ex = raw.copy()

# 1. Fix types: spend -> numeric, date -> datetime
ex["spend"] = pd.to_numeric(ex["spend"], errors="coerce")
ex["date"] = pd.to_datetime(ex["date"], errors="coerce")

# 2. Drop duplicate rows
ex = ex.drop_duplicates()

# 3. Print dtypes + shape
print("Data types:")
print(ex.dtypes)

print("\nFinal shape:")
print(ex.shape)

# 4. Outliers — detect with the IQR **rule**

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])


In [ ]:
# 1. Q1, Q3, and IQR for 'age'
Q1 = df["age"].quantile(0.25)
Q3 = df["age"].quantile(0.75)
IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

# 2. Lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

# 3. Rows outside the bounds
outliers = df[(df["age"] < lower_bound) | (df["age"] > upper_bound)]

print("\nOutlier rows:")
print(outliers)

# 5. Messy text & inconsistent **categories**

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!

In [ ]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

In [ ]:
import pandas as pd

messy = pd.Series(
    [' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
    dtype='string'
)

# 1. Strip whitespace + lowercase
messy = messy.str.strip().str.lower()

# 2. Map 'n.y.' -> 'new york'
messy = messy.replace('n.y.', 'new york')

# 3. value_counts()
print(messy.value_counts())

# **The cleaned dataset**

In [ ]:
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean